# Clash of Clans ML Lab - Problem 3: Clan War Performance Regression

**Objective:** Predict historical clan war performance using **only structural, compositional, infrastructure, and player progression features**.

**Target:** `war_success_rate` (continuous, bounded [0,1])

**Leakage prevention:** Strictly exclude direct war outcome accumulators (`war_wins`, `war_losses`, `war_ties`, `war_win_streak`) from predictor features X.

## 1. Setup & Dynamic Environment

In [ ]:
from pathlib import Path
import sys
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.feature_selection import mutual_info_regression

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 6)

def find_project_root(marker: str = 'data') -> Path:
    """Locate the project root by searching upwards for a directory named `marker`."""
    current = Path.cwd().resolve()
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            return parent
    return current

root = find_project_root()
dataset_path = root / 'data' / 'datasets' / 'clan_war_performance_regression.parquet'
df = pd.read_parquet(dataset_path)

print(f'Project root: {root}')
print(f'Dataset path: {dataset_path}')
print(f'Initial shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

## 2. Data Integrity & Leakage Prevention Audit

In [ ]:
# Dimensions, dtypes, missing values, duplicates
print('--- DataFrame info ---')
df.info(verbose=True, show_counts=True)
print('\n--- Missing values ---')
print(df.isna().sum())
print('\n--- Duplicate rows ---')
print(f'Duplicated rows: {df.duplicated().sum()}')

In [ ]:
# Leakage prevention: direct war history columns must be removed from predictors.
TARGET = 'war_success_rate'
LEAKED_COLS = ['war_wins', 'war_losses', 'war_ties', 'war_win_streak']

leaked_present = [col for col in LEAKED_COLS if col in df.columns]
print(f'Leaked columns present in dataset: {leaked_present}')

if leaked_present:
    print('Dropping leaked columns from feature matrix X.')
    X = df.drop(columns=[TARGET] + leaked_present)
else:
    print('No direct leaked columns found. Proceeding with all non-target columns.')
    X = df.drop(columns=[TARGET])

y = df[TARGET].copy()
print(f'\nX shape after leakage removal: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Any remaining leaked columns in X? {any(col in X.columns for col in LEAKED_COLS)}')

In [ ]:
# Quasi-constant / zero-variance detection (mode frequency >= 95%)
def mode_frequency(series):
    if series.nunique(dropna=True) == 0:
        return 1.0
    if series.nunique(dropna=True) == 1:
        return 1.0
    value_counts = series.value_counts(dropna=True, normalize=True)
    return value_counts.iloc[0] if len(value_counts) > 0 else 1.0

quasi_const_cols = []
for col in X.columns:
    freq = mode_frequency(X[col])
    if freq >= 0.95:
        quasi_const_cols.append((col, freq))

print(f'Quasi-constant columns (mode frequency >= 95%): {len(quasi_const_cols)}')
for col, freq in quasi_const_cols:
    print(f'  {col}: {freq:.4f}')

## 3. Target Analysis (`war_success_rate`)

In [ ]:
target = y
desc = target.describe(percentiles=[.25, .5, .75])
skew = target.skew()
kurt = target.kurtosis()
iqr = desc.loc['75%'] - desc.loc['25%']

print('--- Descriptive statistics for war_success_rate ---')
print(desc)
print(f'\nSkewness: {skew:.4f}')
print(f'Kurtosis: {kurt:.4f}')
print(f'IQR: {iqr:.4f}')
print(f'Min: {desc.loc["min"]:.4f}')
print(f'Max: {desc.loc["max"]:.4f}')

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(target, kde=True, bins=50, color='steelblue', edgecolor='white')
plt.title('Distribution of war_success_rate (Histogram + KDE)')
plt.xlabel('War Success Rate')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

### Target distribution properties and modeling implications

- The target is bounded in [0, 1]. If values accumulate near 0 or 1, the distribution may be skewed or inflated at the extremes.
- **Skewness** and **kurtosis** help decide whether a transform (e.g., logit) or a beta regression is appropriate.
- For standard regression, consider:
  - Using `MinMaxScaler` to keep predictions within [0,1] if using models that do not enforce range (e.g., linear models).
  - Applying a **logit transform** `ln(y / (1-y))` only if no exact 0/1 values exist, otherwise use an epsilon clipping.
  - Using loss functions robust to bounded targets: **Huber loss**, **quantile loss**, or **beta regression**.
- If the distribution is approximately normal, **MSE/RMSE** are natural choices.

## 4. Feature Group Profiling

In [ ]:
# Define feature groups based on column naming patterns.
feature_groups = {
    'Town Hall & Composition': ['TH', 'townhall', 'th'],
    'Player Progression': ['EXP', 'hero_level', 'troop_progress', 'spell_progress', 'equipment_progress'],
    'Infrastructure & Activity': ['clan_level', 'member_count', 'donation_rate', 'capital_contribution_rate', 'clan_capital_points']
}

def assign_group(col):
    col_lower = col.lower()
    for group, patterns in feature_groups.items():
        for pat in patterns:
            if pat.lower() in col_lower:
                return group
    return 'Other'

group_assignments = {col: assign_group(col) for col in X.columns}
X_grouped = X.copy()
X_grouped['__group__'] = X_grouped.columns.map(group_assignments)

print('--- Feature group assignments ---')
for group in feature_groups.keys():
    cols = [col for col, g in group_assignments.items() if g == group]
    print(f'\n{group} ({len(cols)} columns):')
    for c in cols:
        print(f'  - {c}')

In [ ]:
profile_rows = []
for group in feature_groups.keys():
    cols = [col for col, g in group_assignments.items() if g == group]
    if not cols:
        continue
    for col in cols:
        series = X[col]
        profile_rows.append({
            'group': group,
            'feature': col,
            'dtype': str(series.dtype),
            'missing_pct': round(series.isna().mean() * 100, 2),
            'zero_pct': round((series == 0).mean() * 100, 2),
            'skewness': round(series.skew(), 3) if series.dtype in ['float64','int64'] else np.nan
        })

profile_df = pd.DataFrame(profile_rows).sort_values(['group', 'feature'])
print('--- Feature group profiling ---')
display(profile_df)  # works in Jupyter; fallback to print if needed
# If display not available, use print(profile_df.to_string())

## 5. Feature-Target Relationships (Regression)

In [ ]:
# Mutual Information with safe handling of non-finite and missing values.
def safe_mutual_info(X, y, random_state=42):
    X_clean = X.replace([np.inf, -np.inf], np.nan)
    # Impute median for numerical columns (temporary)
    X_imputed = X_clean.copy()
    for col in X_imputed.columns:
        if X_imputed[col].isna().all():
            X_imputed.drop(columns=[col], inplace=True)
            print(f'Dropped column {col}: all values missing.')
        else:
            if X_imputed[col].dtype in ['float64','int64']:
                med = X_imputed[col].median()
                X_imputed[col].fillna(med, inplace=True)
            else:
                X_imputed[col].fillna('MISSING', inplace=True)
    mi = mutual_info_regression(X_imputed, y, random_state=random_state)
    mi_series = pd.Series(mi, index=X_imputed.columns, name='MI')
    return mi_series.sort_values(ascending=False)

mi_series = safe_mutual_info(X, y)
print('--- Mutual Information (top 10) ---')
print(mi_series.head(10).to_string())

In [ ]:
# Pearson and Spearman correlations with the target
pearson_corr = X.corrwith(y, method='pearson').sort_values(ascending=False)
spearman_corr = X.corrwith(y, method='spearman').sort_values(ascending=False)

print('--- Top 10 Pearson ---')
print(pearson_corr.head(10).to_string())
print('\n--- Top 10 Spearman ---')
print(spearman_corr.head(10).to_string())

In [ ]:
# Display combined top tables
mi_top = mi_series.head(10).rename('Mutual Information')
pearson_top = pearson_corr.head(10).rename('Pearson r')
spearman_top = spearman_corr.head(10).rename('Spearman rho')
combined = pd.concat([mi_top, pearson_top, spearman_top], axis=1, join='inner')
print('--- Top feature-target relationship summary (by MI) ---')
display(combined)
# fallback:
# print(combined.to_string())

In [ ]:
top_mi_features = mi_series.head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(top_mi_features):
    ax = axes[i]
    sns.regplot(x=X[col], y=y, ax=ax, scatter_kws={'alpha':0.3, 's':10}, line_kws={'color':'red'})
    ax.set_title(f'Top MI: {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('war_success_rate')
plt.tight_layout()
plt.show()

## 6. Collinearity & Feature Redundancy

In [ ]:
# Correlation matrices on predictors
X_numeric = X.select_dtypes(include=[np.number])
pearson_matrix = X_numeric.corr(method='pearson')
spearman_matrix = X_numeric.corr(method='spearman')

print(f'Pearson matrix shape: {pearson_matrix.shape}')
print(f'Spearman matrix shape: {spearman_matrix.shape}')

In [ ]:
def high_corr_pairs(corr_matrix, threshold=0.85):
    pairs = []
    cols = corr_matrix.columns
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            r = corr_matrix.iloc[i, j]
            if abs(r) > threshold:
                pairs.append((cols[i], cols[j], r))
    return pairs

pearson_high = high_corr_pairs(pearson_matrix)
spearman_high = high_corr_pairs(spearman_matrix)

print(f'High Pearson correlation pairs (|r|>0.85): {len(pearson_high)}')
for pair in pearson_high:
    print(f'  {pair[0]} - {pair[1]}: {pair[2]:.3f}')

print(f'\nHigh Spearman correlation pairs (|ρ|>0.85): {len(spearman_high)}')
for pair in spearman_high:
    print(f'  {pair[0]} - {pair[1]}: {pair[2]:.3f}')

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(pearson_matrix, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            square=True, cbar_kws={'shrink':0.8})
plt.title('Pearson Correlation Matrix of Predictors')
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 10))
sns.heatmap(spearman_matrix, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            square=True, cbar_kws={'shrink':0.8})
plt.title('Spearman Correlation Matrix of Predictors')
plt.tight_layout()
plt.show()

## 7. Outlier & Extreme Case Detection

In [ ]:
def iqr_outlier_percentage(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).mean() * 100

outlier_report = X_numeric.apply(iqr_outlier_percentage).sort_values(ascending=False)
print('--- IQR outlier percentage per feature (top 20) ---')
print(outlier_report.head(20).to_string())

In [ ]:
# Inspect extreme records for selected key features
key_features = ['mean_equipment_progress', 'donation_rate', 'clan_capital_points', 'mean_hero_level']
available_keys = [f for f in key_features if f in X.columns]
print(f'Available key features for extreme inspection: {available_keys}')

for feat in available_keys:
    top_extreme = df.nlargest(10, feat)[[TARGET, feat]]
    print(f'\nTop 10 records for {feat}:')
    print(top_extreme.to_string(index=False))

## 8. Modeling Implications & Executive Summary

In [ ]:
# Construct an executable summary dictionary of findings and recommendations.
summary = {
    'dataset_shape': df.shape,
    'target': TARGET,
    'leakage_columns_removed': leaked_present,
    'quasi_constant_features': [col for col, _ in quasi_const_cols],
    'target_stats': {
        'mean': desc.loc['mean'],
        'std': desc.loc['std'],
        'median': desc.loc['50%'],
        'IQR': iqr,
        'min': desc.loc['min'],
        'max': desc.loc['max'],
        'skewness': skew,
        'kurtosis': kurt
    },
    'top_MI_features': mi_series.head(10).index.tolist(),
    'high_collinearity_pearson_pairs': [(a,b,round(r,3)) for a,b,r in pearson_high],
    'high_collinearity_spearman_pairs': [(a,b,round(r,3)) for a,b,r in spearman_high],
    'preprocessing_recommendations': [
        'Impute missing values using median for numeric features and most frequent category for categorical features.',
        'Apply log1p transformation to highly skewed positive-valued features to reduce skew.',
        'Use RobustScaler for features with outliers to minimize their influence.',
        'Consider MinMaxScaler if using models sensitive to scale (e.g., SVM).',
        'Remove quasi-constant features (mode frequency >= 95%).',
        'For collinear feature pairs (|r|>0.85), retain the feature with higher Mutual Information.'
    ],
    'cv_strategy': (
        'Use StratifiedKFold on binned target (e.g., deciles) to preserve distribution across folds, '
        'or GroupKFold if there is a group identifier to prevent leakage between related clans.'
    ),
    'evaluation_metrics': ['MAE', 'RMSE', 'R^2'],
    'baseline_models': [
        'Ridge Regression (linear baseline, regularized)',
        'Random Forest Regressor (non-linear, robust to outliers)',
        'XGBoost Regressor (gradient boosting, often strong on tabular data)'
    ],
    'feature_selection_strategy': 'Use Mutual Information to rank features and break ties among collinear pairs.',
}

print('--- Executive Summary ---')
for key, value in summary.items():
    if key in ['preprocessing_recommendations', 'baseline_models']:
        print(f'\n{key}:')
        for item in value:
            print(f'  - {item}')
    else:
        print(f'{key}: {value}')

## Final Remarks

This notebook provides a comprehensive EDA for Problem 3. It checks data integrity, enforces leakage prevention, analyzes the target, profiles feature groups, explores feature-target relationships, identifies collinearity and outliers, and outlines modeling recommendations.

The next step is to apply the recommended preprocessing and evaluate the baseline models using the proposed cross-validation strategy.